# Abliterate Qwen3-4B-Thinking-2507 with Heretic

Part of the **say-no-more** experiment. Runs [Heretic](https://github.com/p-e-w/heretic)
on the exact base model used for the baseline eval (pinned commit `768f209d9ea8…`)
and uploads the abliterated result to your Hugging Face account.

### Before you run
1. **Settings → Accelerator → GPU** (T4 ×2).  2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (write access), **Attached** to this notebook.

The Optuna search is checkpointed to `/kaggle/working/checkpoints`, which survives kernel
restarts. If a run finishes the search but stops at the menu, just re-run cell 5 — it resumes
from the checkpoint and goes straight to selecting the trial and uploading (no re-search).

## 1 · GPU check + install
Pins `numpy` to the 2.2 line (what Heretic requires and Kaggle's scipy/numba expect) and
removes Kaggle's old `torchao` (peft rejects torchao <0.16 if present; unused here).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU — enable it in Settings → Accelerator"
%pip install -q "numpy>=2.2,<2.3" heretic-llm
%pip uninstall -y torchao

## 2 · Configuration

In [ ]:
MODEL_ID     = "Qwen/Qwen3-4B-Thinking-2507"
MODEL_COMMIT = "768f209d9ea81521153ed38c47d515654e938aea"   # exact revision used by the baseline
REPO_SUFFIX  = "Qwen3-4B-Thinking-2507-say-no-more"   # -> <your-username>/<this>
PRIVATE      = False   # True to upload as a private repo
N_TRIALS     = 100     # keep the same across resumes so the checkpoint matches
SEED         = 0
CHECKPOINT   = "/kaggle/working/checkpoints"   # Optuna study; persists across restarts

## 3 · Hugging Face login (from the `HF_TOKEN` secret)

In [ ]:
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError(
        "Could not read the HF_TOKEN secret. Add it under Add-ons -> Secrets "
        "(name: HF_TOKEN, a token with WRITE access), and make sure it is "
        "ATTACHED to this notebook."
    ) from None

huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
USERNAME = huggingface_hub.whoami(token=HF_TOKEN)["name"]
REPO_ID  = f"{USERNAME}/{REPO_SUFFIX}"
print("Will upload to:", REPO_ID, "(private)" if PRIVATE else "(public)")

## 4 · Write the abliteration driver

Heretic 1.4.0 drives its menu through `heretic.utils.prompt_select` / `prompt_text`
(plain `input()` off a TTY), **not** questionary. This script patches those functions so every
prompt is answered automatically: resume the checkpoint → **best trial** (lowest refusals) →
**upload** → exit. The repo name, visibility and reproducibility are preset, and the token comes
from the environment. It reads settings from env vars so the next cell can run it in a clean
subprocess.

In [ ]:
%%writefile /kaggle/working/run_heretic.py
import os, sys, huggingface_hub

MODEL_ID     = os.environ["MODEL_ID"]
MODEL_COMMIT = os.environ.get("MODEL_COMMIT") or None
REPO_ID      = os.environ["REPO_ID"]
PRIVATE      = os.environ.get("PRIVATE", "0") == "1"
N_TRIALS     = os.environ.get("N_TRIALS", "100")
SEED         = os.environ.get("SEED", "0")
CHECKPOINT   = os.environ.get("CHECKPOINT", "/kaggle/working/checkpoints")
HF_TOKEN     = os.environ["HF_TOKEN"]

huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
huggingface_hub.get_token = lambda: HF_TOKEN   # Heretic reads the token from here

import heretic.main as hmain
import heretic.utils as hutils

_counts = {}
def _disp(c):
    if isinstance(c, str):
        return c
    t = getattr(c, "title", None)
    return t if isinstance(t, str) else str(getattr(c, "value", c))
def _val(c):
    if isinstance(c, str):
        return c
    return getattr(c, "value", c)
def _find(choices, needle):
    for c in choices:
        if needle in _disp(c).lower():
            return _val(c)
    return None

def patched_select(message, choices):
    m = str(message).lower()
    _counts[m] = _counts.get(m, 0) + 1
    if "how would you like to proceed" in m:      # checkpoint menu
        return (_find(choices, "show the results") or _find(choices, "continue")
                or _val(choices[0]))
    if "which trial" in m:
        if _counts[m] == 1:
            return _val(choices[0])               # best: fewest refusals
        return _find(choices, "exit") or _val(choices[-1])   # 2nd visit -> exit
    if "what do you want to do" in m:
        return _find(choices, "upload")
    if "export" in m:
        return _find(choices, "merge")
    if "public or private" in m:
        return _find(choices, "private" if PRIVATE else "public")
    if "reproducibility" in m:
        return _find(choices, "full")
    return _val(choices[0])

def patched_text(message, default="", qmark="?", unsafe=False):
    if "repository" in str(message).lower():
        return REPO_ID
    return default

def patched_path(message):
    return "/kaggle/working/out"

def patched_password(message):
    return HF_TOKEN

for mod in (hmain, hutils):
    mod.prompt_select   = patched_select
    mod.prompt_text     = patched_text
    mod.prompt_path     = patched_path
    mod.prompt_password = patched_password

sys.argv = ["heretic", "--model", MODEL_ID,
            "--export-strategy", "merge", "--n-trials", N_TRIALS, "--seed", SEED,
            "--study-checkpoint-dir", CHECKPOINT]
if MODEL_COMMIT:
    sys.argv += ["--model-commit", MODEL_COMMIT]

hmain.run()
print("DONE: abliterated model uploaded to https://huggingface.co/" + REPO_ID)

## 5 · Run it (fresh subprocess)

Runs in a fresh `python` process (clean numpy import) with cwd `/kaggle/working` so the
default/explicit checkpoint path lines up. If the checkpoint from a previous run exists, this
resumes to the trial menu and uploads in minutes; otherwise it runs the ~3-hour search first.

In [ ]:
import os, sys, subprocess
env = {**os.environ,
       "MODEL_ID": MODEL_ID, "MODEL_COMMIT": MODEL_COMMIT, "REPO_ID": REPO_ID,
       "PRIVATE": "1" if PRIVATE else "0", "N_TRIALS": str(N_TRIALS),
       "SEED": str(SEED), "CHECKPOINT": CHECKPOINT, "HF_TOKEN": HF_TOKEN}
proc = subprocess.run([sys.executable, "/kaggle/working/run_heretic.py"],
                      env=env, cwd="/kaggle/working")
if proc.returncode != 0:
    raise SystemExit(f"Heretic exited with code {proc.returncode}")

## 6 · Verify the upload

In [ ]:
from huggingface_hub import list_repo_files
print("Uploaded files:")
for f in sorted(list_repo_files(REPO_ID, token=HF_TOKEN)):
    print("  ", f)
print("\nModel: https://huggingface.co/" + REPO_ID)
print("\nNext, on the Mac:")
print(f"  .venv/bin/hf download {REPO_ID} --local-dir models/{REPO_SUFFIX}")
print(f"  .venv/bin/mlx_lm.convert --hf-path models/{REPO_SUFFIX} \\")
print(f"      --mlx-path models/{REPO_SUFFIX}-mlx-8bit -q --q-bits 8")
print(f"  MODEL=models/{REPO_SUFFIX}-mlx-8bit scripts/serve.sh   # then scripts/eval.sh 03-abliterated")